# Document Summarizer Example

This notebook demonstrates how to use the book_summarizer package to:
1. Load and manage documents
2. Process highlights
3. Generate AI-powered summaries

In [1]:

import nest_asyncio
nest_asyncio.apply()
from book_summarizer import (
    DocumentHandler, 
    HighlightProcessor, 
    SummaryGenerator, 
    Config
)
from tinydb import TinyDB
import os, json

# Initialize configuration
Config.initialize()

# Initialize databases
main_db = TinyDB(Config.MAIN_DB_PATH)
highlights_db = TinyDB(Config.HIGHLIGHTS_DB_PATH)

# Initialize components
doc_handler = DocumentHandler(main_db, highlights_db)
highlight_processor = HighlightProcessor(highlights_db)
summary_generator = SummaryGenerator()

## View Available Documents

Let's see what documents are already in the system

In [2]:
docs = doc_handler.list_documents()

print("Available Documents:")
print("-" * 50)
for doc in docs:
    print(f"ID: {doc.uid}")
    print(f"Name: {doc.name}")
    print(f"Folder: {doc.folder}")
    print("-" * 50)

Available Documents:
--------------------------------------------------
ID: the-invisible-empire
Name: The Invisible Empire
Folder: ./data/srcs/the-invisible-empire/
--------------------------------------------------


In [2]:
# get document details
doc_id = "the-invisible-empire"
doc = doc_handler.get_document(doc_id)
doc_dict = doc.__dict__ if doc else {}  # Convert Book object to dictionary
print(json.dumps(doc_dict, indent=2))




{
  "uid": "the-invisible-empire",
  "name": "The Invisible Empire",
  "folder": "./data/srcs/the-invisible-empire/",
  "toc": [
    {
      "type": "link",
      "href": "xhtml/cover.xhtml",
      "title": "Cover",
      "uid": "cover"
    },
    {
      "type": "link",
      "href": "xhtml/toc.xhtml",
      "title": "Contents",
      "uid": "html-toc"
    },
    {
      "type": "link",
      "href": "xhtml/c001.xhtml",
      "title": "1 BOUNTY",
      "uid": "c001"
    },
    {
      "type": "link",
      "href": "xhtml/c002.xhtml",
      "title": "2 A WHOLE NEW WORLD",
      "uid": "c002"
    },
    {
      "type": "link",
      "href": "xhtml/c003.xhtml",
      "title": "3 SUPERSIZE ME",
      "uid": "c003"
    },
    {
      "type": "link",
      "href": "xhtml/c004.xhtml",
      "title": "4 THE VIRUS IS US",
      "uid": "c004"
    },
    {
      "type": "link",
      "href": "xhtml/c005.xhtml",
      "title": "5 A DEEP CONTROL",
      "uid": "c005"
    },
    {
      "type": "li

## Add Highlights

If you have highlights available, you can add them to the document

In [3]:
# View highlights for a specific chapter
chapter_highlights = highlight_processor.find_highlights_for_chapter(
    chapter_title="1 BOUNTY",
    doc_id=doc_id
)

print("Highlights for 1 BOUNTY:")
for highlight in chapter_highlights:
    print(f"- {highlight}")

Highlights for 1 BOUNTY:
- A single gram of the stale-smelling yellow grimy film on our teeth, good old plaque, has approximately 1011 bacteria, which is about the same number as that of all the humans that have ever lived.
- The effect of this interdependence is deeply significant as it sets the tone of relationships for all life on   Earth.
- The problem with viruses is that they do not fit into any of the conventionally accepted domains of life— they are neither archaea, eukaryotes nor prokaryotes.


## Generate Summaries

Now let's generate summaries for each chapter

In [3]:
def save_summary_to_markdown(
    chapter_title:str,
    summary: any, 
    output_path: str
) -> None:
    """
    Save the chapter summary to a markdown file.
    
    Args:
        summary: ChapterSummary object containing the summary
        output_path: Path where the markdown file should be saved
    """
    with open(output_path, 'a') as f:
        f.write(f"## {chapter_title}\n")
        f.write(f"_{summary.chapter_descriptor}_\n\n")
        f.write(f"{summary.chapter_summary}\n")
        f.write(f"### Concept Map\n")
        f.write(f"```mermaid\n")
        f.write(f"{summary.mermaid_graph}\n")
        f.write(f"```\n")
        f.write(f"### Themes & Ideas\n")
        
        for info_point in summary.info_points:
            f.write(f"#### {info_point.subtitle}\n")
            for pointer in info_point.pointers:
                f.write(f"* {pointer.pointer}\n")
                if pointer.related_quote_highlighted_by_reader:
                    f.write(f"  > [!quote] {pointer.related_quote_highlighted_by_reader}\n") 
        f.write("---\n\n")

In [4]:
# Get document details
doc = doc_handler.get_document(doc_id)
if not doc:
    print(f"Document not found: {doc_id}")
else:
    book_name = "The Invisible Empire"
    output_file = f"../storage/notebook/Book Notes/{book_name}.md"
    # Check if file exists
    if os.path.exists(output_file):
        user_input = input(f"File {output_file} already exists. Do you want to empty it? (y/n): ")
        if user_input.lower() == 'y':
            # Empty the file
            open(output_file, 'w').close()
    else:
        # Create the directory if it doesn't exist
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        # Create empty file
        open(output_file, 'w').close()
    

        
    for chapter in doc.toc:
        if chapter['type'] != 'link' or chapter['title'] in ['Cover', 'Contents','Copyright','Follow Penguin','Acknowledgements','List of Illustrations','Notes']:
            continue
            
        # Get chapter content
        chapter_path = os.path.join(
            Config.WORKING_BASE_DIR,
            doc_id,
            "unbundled_epub",
            chapter['href']
        )
        
        chapter_content = summary_generator.extract_clean_content(chapter_path)
        if not chapter_content:
            continue

        # Get highlights for this chapter
        highlights = highlight_processor.find_highlights_for_chapter(
            chapter['title'],
            doc_id
        )
        # print the number of highlights found
        print(f"Found {len(highlights)} highlights for chapter: {chapter['title']}")
        

        print(f"Processing chapter: {chapter['title']}")
        
        # Generate summary
        summary = summary_generator.generate_chapter_summary(
            chapter_content,
            highlights
        )

        summary_data = summary.data

        # Write chapter summary to the file using the save_summary_to_markdown function
        save_summary_to_markdown(chapter['title'],summary_data, output_file)

        print(f"Added summary for chapter: {chapter['title']}")

    print(f"Complete summary saved to: {output_file}")

Found 3 highlights for chapter: 1 BOUNTY
Processing chapter: 1 BOUNTY
Added summary for chapter: 1 BOUNTY
Found 0 highlights for chapter: 2 A WHOLE NEW WORLD
Processing chapter: 2 A WHOLE NEW WORLD
Added summary for chapter: 2 A WHOLE NEW WORLD
Found 0 highlights for chapter: 3 SUPERSIZE ME
Processing chapter: 3 SUPERSIZE ME
Added summary for chapter: 3 SUPERSIZE ME
Found 10 highlights for chapter: 4 THE VIRUS IS US
Processing chapter: 4 THE VIRUS IS US
Added summary for chapter: 4 THE VIRUS IS US
Found 9 highlights for chapter: 5 A DEEP CONTROL
Processing chapter: 5 A DEEP CONTROL
Added summary for chapter: 5 A DEEP CONTROL
Found 1 highlights for chapter: 6 INVADERS, HITCH-HIKERS, SENTINELS, KILLERS
Processing chapter: 6 INVADERS, HITCH-HIKERS, SENTINELS, KILLERS
Added summary for chapter: 6 INVADERS, HITCH-HIKERS, SENTINELS, KILLERS
Found 2 highlights for chapter: 7 A SPOTTY HISTORY OF THE SPECKLED MONSTER
Processing chapter: 7 A SPOTTY HISTORY OF THE SPECKLED MONSTER
Added summary f

In [6]:
print(summary)

RunResult(_all_messages=[ModelRequest(parts=[UserPromptPart(content="\n        Below is a content of a chapter from a book along with quotes highlighted by the reader.\n        \n        Book Content:\n        \n\nInvisible Empire: The Natural History Of Viruses\n\n\n\n\n\n3\nSUPERSIZE ME\nIt is widely agreed that major evolutionary leaps—including the emergence of life itself—occurred in deep water or at the water’s edge. Experts tell us that the first cellular life was jump-started in a tidal pool or in a deep ocean vent where energy and nutrients were mixed. And wherever there are cells and microbes, there are viruses! Water is teeming with an unfathomable abundance of these—from the salt pans of Qatar, the hot springs of New Zealand, Bengal’s tidal mangroves, deep-sea vents in the North Atlantic Ridge, under an ice shelf in Antarctica or in a sewage drain near you—the diversity of viruses is outmatched only by their sheer numbers.\nIn 1992, Timothy Rowbotham, a microbiologist with 